# Исследовательский ноутбук прогнозированиz депозитов населения Беларуси
Помогают ли данные о новых депозитных договорах точнее прогнозировать остатки на счетах населения? Проверим это на статистике НБРБ: сравним простые прогнозы с Ridge, затем добавим показатели новых договоров и регуляторных условий. Рублёвые и валютные депозиты рассмотрим отдельно.
Есть важная оговорка: новый договор — не обязательно новые деньги. В статистику входят и продления вкладов.
Используем остатки за декабрь 2014 — август 2026 года, новые договоры — с января 2020 года. Этот период уже изучался при разработке моделей; здесь мы воспроизводим расчёты. Независимая проверка потребует новых данных.

In [10]:
from pathlib import Path
import pandas as pd

In [27]:
file_path = Path("data/broadmoney_ru.xlsx")
print("Файл найден:", file_path.exists())
excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

Файл найден: True


['2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020',
 '2021',
 '2022',
 '2023',
 '2024',
 '2025',
 '2026']

In [28]:
df = pd.read_excel(excel_file, sheet_name='2016', header=None)
df.head(15)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Широкая денежная масса за 2016 год,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,млн. рублей,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Показатели,2016-01-01 00:00:00,2016-02-01 00:00:00,2016-03-01 00:00:00,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,2016-09-01 00:00:00,2016-10-01 00:00:00,2016-11-01 00:00:00,2016-12-01 00:00:00
4,1. Наличные деньги в обороте – М0,1423.6082,1343.8813,1355.5896,1364.9471,1441.6253,1428.8349,1692.7634,1562.451,1542.91,1608.642,1625.645,1642.95
5,2. Переводные депозиты,2849.7181,2388.4144,2608.5468,2489.4005,2627.5133,2676.3675,2566.8589,2779.727,2834.008,2949.729,2866.11,3558.32
6,2.1. Физических лиц,1405.9848,1288.0616,1235.0413,1179.5471,1349.718,1238.6432,1138.7313,1434.5,1305.086,1424.209,1322.598,1373.225
7,2.2. Юридических лиц^,1443.7333,1100.3528,1373.5055,1309.8534,1277.7953,1437.7243,1428.1276,1345.227,1528.922,1525.52,1543.512,2185.095
8,Денежный агрегат – М1,4273.3263,3732.2957,3964.1364,3854.3476,4069.1386,4105.2024,4259.6223,4342.178,4376.918,4558.371,4491.755,5201.27
9,3. Другие депозиты,4640.23098,4249.30077,4147.134509,4438.498502,4234.005366,4601.724577,4839.562459,4555.1574,4950.356,5080.40886,5011.25823,5356.59678


In [30]:
print(df.iloc[:, 0].to_string())

0                    Широкая денежная масса за 2016 год
1                                                   NaN
2                                           млн. рублей
3                                            Показатели
4                     1. Наличные деньги в обороте – М0
5                                2. Переводные депозиты
6                                   2.1. Физических лиц
7                                 2.2. Юридических лиц^
8                                 Денежный агрегат – М1
9                                    3. Другие депозиты
10                                  3.1. Физических лиц
11                                3.2. Юридических лиц^
12       Денежная масса в национальном определении – М2
13    4. Ценные бумаги, выпущенные банками (вне банк...
14                        Рублевая денежная масса – М2*
15                     5. Депозиты в иностранной валюте
16                             5.1. Переводные депозиты
17                                5.1.1. Физичес

In [47]:
stocks_2016 = df.iloc[[6, 10, 29, 32], 1:].copy()
stocks_2016 = stocks_2016.T
stocks_2016.columns = ['byn_transfer', 'byn_other', 'fx_transfer_usd', 'fx_other_usd']

dates = df.iloc[3, 1:].values

stocks_2016.index = dates

stocks_2016.head()

,byn_transfer,byn_other,fx_transfer_usd,fx_other_usd
2016-01-01,1405.9848,2614.4611,521.315,7691.459
2016-02-01,1288.0616,2465.5978,419.011,7665.671
2016-03-01,1235.0413,2428.7039,391.668,7549.098
2016-04-01,1179.5471,2471.2406,376.209,7538.571
2016-05-01,1349.718,2494.9795,388.624,7445.861


In [68]:
raw = pd.read_excel(file_path, sheet_name="2016", header=None)


stocks_2016 = raw.iloc[[6, 10, 29, 32], 1:].T.copy()
stocks_2016.columns = [
    "byn_transfer",
    "byn_other",
    "fx_transfer_usd",
    "fx_other_usd",
]

stocks_2016 = stocks_2016.apply(pd.to_numeric, errors="raise")


dates = pd.to_datetime(raw.iloc[3, 1:].values)
stocks_2016.index = dates.to_period("M") - 1
stocks_2016.index.name = "month"

stocks_2016["byn_stock_mln"] = (
    stocks_2016["byn_transfer"] + stocks_2016["byn_other"]
)
stocks_2016["fx_stock_usd_mln"] = (
    stocks_2016["fx_transfer_usd"] + stocks_2016["fx_other_usd"]
)

display(stocks_2016)
print(stocks_2016.dtypes)
print("Пропуски:")
print(stocks_2016.isna().sum())

,byn_transfer,byn_other,fx_transfer_usd,fx_other_usd,byn_stock_mln,fx_stock_usd_mln
month,,,,,,
2015-12,1405.9848,2614.4611,521.315,7691.459,4020.4459,8212.774
2016-01,1288.0616,2465.5978,419.011,7665.671,3753.6594,8084.682
2016-02,1235.0413,2428.7039,391.668,7549.098,3663.7452,7940.766
2016-03,1179.5471,2471.2406,376.209,7538.571,3650.7877,7914.780
2016-04,1349.7180,2494.9795,388.624,7445.861,3844.6975,7834.485
2016-05,1238.6432,2525.3127,395.121,7337.839,3763.9559,7732.960
2016-06,1138.7313,2537.6569,412.500,7267.381,3676.3882,7679.881
2016-07,1434.5000,2595.7720,415.530,7236.531,4030.2720,7652.061
2016-08,1305.0860,2661.0640,414.977,7222.993,3966.1500,7637.970


byn_transfer        float64
byn_other           float64
fx_transfer_usd     float64
fx_other_usd        float64
byn_stock_mln       float64
fx_stock_usd_mln    float64
dtype: object
Пропуски:
byn_transfer        0
byn_other           0
fx_transfer_usd     0
fx_other_usd        0
byn_stock_mln       0
fx_stock_usd_mln    0
dtype: int64


In [67]:
def read_stock_year(excel_file, year):
    raw = pd.read_excel(excel_file, sheet_name=str(year), header=None)

    stocks = raw.iloc[[6, 10, 29, 32], 1:].T.copy()
    stocks.columns = [
        "byn_transfer",
        "byn_other",
        "fx_transfer_usd",
        "fx_other_usd",
    ]
    stocks = stocks.apply(pd.to_numeric, errors="raise")

    dates = pd.to_datetime(raw.iloc[3, 1:].values)
    stocks.index = dates.to_period("M") - 1
    stocks.index.name = "month"

    stocks["byn_stock_mln"] = (
        stocks["byn_transfer"] + stocks["byn_other"]
    )
    stocks["fx_stock_usd_mln"] = (
        stocks["fx_transfer_usd"] + stocks["fx_other_usd"]
    )

    return stocks

PeriodIndex(['2015-12', '2016-01', '2016-02', '2016-03', '2016-04', '2016-05',
             '2016-06', '2016-07', '2016-08', '2016-09', '2016-10', '2016-11'],
            dtype='period[M]', name='month')
byn_transfer        object
byn_other           object
fx_transfer_usd     object
fx_other_usd        object
byn_stock_mln       object
fx_stock_usd_mln    object
dtype: object
byn_transfer        0
byn_other           0
fx_transfer_usd     0
fx_other_usd        0
byn_stock_mln       0
fx_stock_usd_mln    0
dtype: int64
